# Standalone Kaggle Qwen2.5 3B GRPO Training with Feedback

This is a **fully self-contained** notebook — every dependency from `oncallenv`, the simulator, the reward system, and the training harness is inlined below. No repo clone needed.

## 1. GPU Check

In [18]:
!nvidia-smi

Sun Apr 26 08:22:25 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.105.08             Driver Version: 580.105.08     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   63C    P0             29W /   70W |    3301MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 2. Install Dependencies

Only pip packages — no repo clone required.

In [19]:
!pip install -U pip setuptools wheel
!pip install torch transformers datasets accelerate trl peft bitsandbytes unsloth matplotlib seaborn pydantic pyyaml openai tqdm

## 3. Inlined Simulator & Environment Code

Everything from `oncallenv` (graph, faults, telemetry, tools, types, env, rewards, curriculum) is defined here.

In [20]:
# ============================================================================
# simulation/graph.py — Microservice graph state used by the simulator
# ============================================================================
from __future__ import annotations
from dataclasses import dataclass, field
from typing import Any, Optional, Literal, Dict, List, Tuple, Iterator, Callable
import hashlib, json, os, random, re, time, inspect
from abc import ABC, abstractmethod
from pathlib import Path
from collections import defaultdict

from pydantic import BaseModel, Field, ConfigDict, ValidationError


@dataclass
class Service:
    name: str
    cpu: float = 25.0
    memory: float = 40.0
    latency_p50: float = 12.0
    latency_p99: float = 45.0
    error_rate: float = 0.001
    rps: float = 500.0
    dependencies: list[str] = field(default_factory=list)
    config: dict[str, Any] = field(default_factory=dict)
    deploy_history: list[dict[str, Any]] = field(default_factory=list)
    logs: list[str] = field(default_factory=list)
    current_fault: str | None = None
    remediated: bool = False


class MicroserviceGraph:
    def __init__(self, services: dict[str, Service]):
        self.services = services
        self.elapsed_sec = 0
        self.event_log: list[dict[str, str]] = []
        self.root_cause_service = ""
        self.root_cause_category = ""
        self.required_remediations: set[str] = set()
        self.completed_remediations: set[str] = set()

    @classmethod
    def topology(cls, name: str) -> "MicroserviceGraph":
        base = {
            "api-gateway": Service("api-gateway", dependencies=["checkout-service", "user-service"]),
            "checkout-service": Service("checkout-service", dependencies=["payment-service", "inventory-service"]),
            "payment-service": Service("payment-service", dependencies=["postgres-primary", "redis-cache"]),
            "inventory-service": Service("inventory-service", dependencies=["postgres-primary"]),
            "user-service": Service("user-service", dependencies=["postgres-primary"]),
            "postgres-primary": Service("postgres-primary", cpu=35.0, memory=55.0, rps=1800.0),
            "redis-cache": Service("redis-cache", cpu=18.0, memory=35.0, rps=2600.0),
        }
        if name == "deep_chain":
            base["api-gateway"].dependencies = ["checkout-service"]
            base["checkout-service"].dependencies = ["payment-service"]
            base["payment-service"].dependencies = ["inventory-service"]
            base["inventory-service"].dependencies = ["postgres-primary"]
        elif name == "star":
            for svc in base.values():
                if svc.name != "api-gateway":
                    svc.dependencies = ["api-gateway"]
        elif name == "mesh":
            base["checkout-service"].dependencies.append("user-service")
            base["payment-service"].dependencies.append("inventory-service")
            base["inventory-service"].dependencies.append("redis-cache")
        elif name == "bipartite":
            base["api-gateway"].dependencies = ["checkout-service", "payment-service", "inventory-service"]
            base["checkout-service"].dependencies = ["postgres-primary", "redis-cache"]
            base["payment-service"].dependencies = ["postgres-primary", "redis-cache"]
            base["inventory-service"].dependencies = ["postgres-primary", "redis-cache"]
        elif name == "diamond":
            base["api-gateway"].dependencies = ["checkout-service", "user-service"]
            base["checkout-service"].dependencies = ["payment-service"]
            base["user-service"].dependencies = ["payment-service"]
            base["payment-service"].dependencies = ["postgres-primary"]
        return cls(base)

    def tick(self, seconds: int = 30) -> None:
        self.elapsed_sec += seconds
        for svc in self.services.values():
            if svc.current_fault and not svc.remediated:
                svc.latency_p99 *= 1.03
                svc.error_rate = min(0.99, svc.error_rate + 0.015)

    def service_names(self) -> list[str]:
        return list(self.services)

    def get(self, service_name: str) -> Service:
        if service_name not in self.services:
            raise KeyError(f"unknown service {service_name}")
        return self.services[service_name]

    def mark_remediated(self, action_key: str, service_name: str) -> bool:
        key = f"{action_key}:{service_name}"
        if key not in self.required_remediations:
            return False
        self.completed_remediations.add(key)
        svc = self.services[service_name]
        svc.remediated = True
        svc.current_fault = None
        svc.cpu = min(svc.cpu, 30.0)
        svc.memory = min(svc.memory, 45.0)
        svc.latency_p50 = min(svc.latency_p50, 25.0)
        svc.latency_p99 = min(svc.latency_p99, 95.0)
        svc.error_rate = 0.001
        return self.is_recovered()

    def is_recovered(self) -> bool:
        return bool(self.required_remediations) and self.required_remediations.issubset(self.completed_remediations)

    def synthetic_success_rate(self) -> float:
        affected = self.services.get(self.root_cause_service)
        if affected is None:
            return 0.0
        if self.is_recovered():
            return 1.0
        return max(0.0, 1.0 - affected.error_rate)

    def blast_radius_score(self) -> float:
        affected = self.services.get(self.root_cause_service)
        if affected is None:
            return 0.0
        exposure = min(1.0, self.elapsed_sec / 900.0)
        return max(0.0, 1.0 - (affected.error_rate * 0.65 + exposure * 0.35))


print("[OK] MicroserviceGraph defined")

[OK] MicroserviceGraph defined


In [21]:
# ============================================================================
# simulation/faults.py — Fault primitives for the Red Shift simulator
# ============================================================================

@dataclass(frozen=True)
class FaultPrimitive:
    name: str
    log_lines: tuple[str, ...]
    cpu: float
    memory: float
    latency_p99: float
    error_rate: float
    remediation: str

    def apply(self, graph: MicroserviceGraph, service_name: str) -> None:
        svc = graph.get(service_name)
        svc.current_fault = self.name
        svc.cpu = max(svc.cpu, self.cpu)
        svc.memory = max(svc.memory, self.memory)
        svc.latency_p50 = max(svc.latency_p50, self.latency_p99 / 5)
        svc.latency_p99 = max(svc.latency_p99, self.latency_p99)
        svc.error_rate = max(svc.error_rate, self.error_rate)
        svc.logs.extend(self.log_lines)
        svc.deploy_history.append({"version": "v3.2.1", "date": "2026-04-24", "status": "current", "notes": self.name})
        graph.root_cause_service = service_name
        graph.root_cause_category = self.name
        graph.required_remediations.add(f"{self.remediation}:{service_name}")
        graph.event_log.append({"timestamp": "2026-04-24T09:00:00Z", "service": service_name, "description": f"{self.name} injected into {service_name}"})


FAULTS: dict[str, FaultPrimitive] = {
    "oom_kill": FaultPrimitive("oom_kill", ("java.lang.OutOfMemoryError: Java heap space", "Container killed by OOMKilled / exit code 137"), 55.0, 94.0, 4200.0, 0.22, "kubectl_rollout_restart"),
    "cpu_hog": FaultPrimitive("cpu_hog", ("run queue saturated", "CPU throttling at 98 percent"), 98.0, 50.0, 2500.0, 0.16, "kubectl_scale"),
    "network_partition": FaultPrimitive("network_partition", ("context deadline exceeded", "Envoy response flag UF upstream failure"), 35.0, 45.0, 5000.0, 0.35, "traffic_split_update"),
    "dns_misconfig": FaultPrimitive("dns_misconfig", ("lookup inventory.internal: no such host", "Envoy response flag NR no route"), 30.0, 40.0, 3600.0, 0.28, "kubectl_apply_config"),
    "replica_lag": FaultPrimitive("replica_lag", ("replication_lag_seconds above threshold", "stale read detected"), 70.0, 72.0, 2100.0, 0.11, "feature_flag_toggle"),
    "cache_stampede": FaultPrimitive("cache_stampede", ("redis miss rate 96 percent", "thundering herd after cache expiry"), 88.0, 76.0, 3100.0, 0.18, "feature_flag_toggle"),
    "http_503_loop": FaultPrimitive("http_503_loop", ("upstream returned HTTP 503", "Envoy response flag UH no healthy upstream"), 66.0, 58.0, 3900.0, 0.41, "kubectl_rollout_undo"),
    "deadlock": FaultPrimitive("deadlock", ("deadlock detected while waiting for lock", "org.postgresql.util.PSQLException: Cannot get connection"), 62.0, 68.0, 4500.0, 0.25, "kubectl_rollout_restart"),
    "disk_full": FaultPrimitive("disk_full", ("no space left on device", "etcdserver: data corruption detected"), 50.0, 60.0, 2400.0, 0.21, "kubectl_apply_config"),
    "cert_expiry": FaultPrimitive("cert_expiry", ("x509: certificate has expired or is not yet valid", "TLS handshake failed"), 25.0, 40.0, 5000.0, 0.50, "kubectl_apply_config"),
    "clock_skew": FaultPrimitive("clock_skew", ("JWT not valid yet due to clock skew", "signature timestamp outside tolerance"), 28.0, 35.0, 1800.0, 0.19, "kubectl_rollout_restart"),
    "gc_pause": FaultPrimitive("gc_pause", ("GC overhead limit exceeded", "called Result::unwrap() on an Err value"), 75.0, 90.0, 4700.0, 0.20, "kubectl_rollout_restart"),
}

print(f"[OK] {len(FAULTS)} fault primitives defined")

[OK] 12 fault primitives defined


In [22]:
# ============================================================================
# telemetry/otlp.py — OpenTelemetry-shaped metrics, logs, and traces
# ============================================================================

def prometheus_metrics(graph: MicroserviceGraph, service_name: str) -> str:
    svc = graph.get(service_name)
    labels = f'service="{svc.name}"'
    return "\n".join([
        f'http_requests_total{{{labels},status="200"}} {int(svc.rps * (1 - svc.error_rate))}',
        f'http_requests_total{{{labels},status="500"}} {int(svc.rps * svc.error_rate)}',
        f'http_request_duration_seconds{{{labels},quantile="0.50"}} {svc.latency_p50 / 1000:.3f}',
        f'http_request_duration_seconds{{{labels},quantile="0.99"}} {svc.latency_p99 / 1000:.3f}',
        f'process_cpu_usage{{{labels}}} {svc.cpu / 100:.3f}',
        f'process_resident_memory_ratio{{{labels}}} {svc.memory / 100:.3f}',
    ])


def json_logs(graph: MicroserviceGraph, service_name: str) -> str:
    svc = graph.get(service_name)
    rows = []
    for idx, body in enumerate(svc.logs[-20:], start=1):
        digest = hashlib.sha1(f"{svc.name}:{idx}:{body}".encode()).hexdigest()
        rows.append({
            "timestamp": f"2026-04-24T09:{idx:02d}:00Z",
            "trace_id": digest[:32], "span_id": digest[32:48],
            "service.name": svc.name,
            "severity": "ERROR" if any(t in body.lower() for t in ["error", "expired", "killed", "deadline", "503"]) else "WARN",
            "body": body,
            "attributes": {"fault": svc.current_fault or "none"},
        })
    return "\n".join(json.dumps(r, sort_keys=True) for r in rows) or "no recent logs"


def jaeger_traces(graph: MicroserviceGraph, service_name: str) -> str:
    svc = graph.get(service_name)
    spans, parent = [], None
    for dep in [svc.name, *svc.dependencies]:
        span_id = hashlib.md5(dep.encode()).hexdigest()[:16]
        spans.append({"service": dep, "span_id": span_id, "parent_span_id": parent, "duration_ms": graph.get(dep).latency_p99 if dep in graph.services else 20})
        parent = span_id
    return json.dumps({"data": [{"traceID": hashlib.md5(svc.name.encode()).hexdigest(), "spans": spans}]}, indent=2)


print("[OK] Telemetry functions defined")

[OK] Telemetry functions defined


In [23]:
# ============================================================================
# core/types.py — Pydantic models (standalone, no openenv dependency)
# ============================================================================

FaultName = Literal[
    "oom_kill", "cpu_hog", "network_partition", "dns_misconfig", "replica_lag",
    "cache_stampede", "http_503_loop", "deadlock", "disk_full", "cert_expiry",
    "clock_skew", "gc_pause",
]


class ScenarioSpec(BaseModel):
    task_id: str
    topology: Literal["simple_fanout", "deep_chain", "mesh", "star", "bipartite", "diamond"]
    fault_primary: FaultName
    fault_secondary: Optional[FaultName] = None
    inject_service: str
    latency_ms: int
    blast_radius: float = Field(ge=0.0, le=1.0)
    metric_noise: float = Field(ge=0.0, le=1.0)
    red_herring: Optional[Literal[
        "unrelated_alert", "stale_deploy_notice", "innocent_config_change",
        "flapping_canary", "false_correlation", "old_anomaly", "unused_service_spike",
    ]] = None
    deploy_window: Optional[Literal["recent_deploy", "flag_flip", "config_change", "none"]] = "none"
    schema_drift: Optional[Literal[
        "rename_metric", "swap_units", "rotate_creds", "new_required_field",
        "field_type_change", "endpoint_version_bump", "none",
    ]] = "none"
    seed: int
    max_steps: int = 25


class Alert(BaseModel):
    alert_id: str
    severity: Literal["critical", "warning", "info"]
    service: str
    message: str
    timestamp: str


class Citation(BaseModel):
    source: Literal["log", "metric", "trace", "config"]
    ref: str
    excerpt: str


class TimelineEvent(BaseModel):
    timestamp: str
    service: str
    description: str


class RCA(BaseModel):
    root_cause_service: str
    root_cause_category: str
    timeline: list[TimelineEvent] = Field(default_factory=list)
    five_whys: list[str] = Field(default_factory=list)
    action_items: list[str] = Field(default_factory=list)
    evidence_citations: list[Citation] = Field(default_factory=list)
    blast_radius_description: str = ""


class Reward(BaseModel):
    total: float
    breakdown: dict[str, float] = Field(default_factory=dict)


class Observation(BaseModel):
    model_config = ConfigDict(extra="forbid", validate_assignment=True, arbitrary_types_allowed=True)
    done: bool = False
    reward: float | int | None = None
    metadata: dict[str, Any] = Field(default_factory=dict)
    alerts: list[Alert] = Field(default_factory=list)
    last_action_result: str = ""
    available_tools: list[str] = Field(default_factory=list)
    services: list[str] = Field(default_factory=list)
    time_elapsed_sec: int = 0
    goal: str = ""
    rca_required: bool = True
    reward_breakdown: dict[str, float] = Field(default_factory=dict)
    task_id: str = ""


class Action(BaseModel):
    model_config = ConfigDict(extra="forbid", validate_assignment=True, arbitrary_types_allowed=True)
    command: str
    metadata: dict[str, Any] = Field(default_factory=dict)


class State(BaseModel):
    model_config = ConfigDict(extra="allow", validate_assignment=True, arbitrary_types_allowed=True)
    episode_id: Optional[str] = None
    step_count: int = Field(default=0, ge=0)
    task_id: str = ""
    done: bool = False
    actions_taken: list[str] = Field(default_factory=list)
    resolved_declared: bool = False
    rca: Optional[RCA] = None
    last_action_result: str = ""
    reward_breakdown: dict[str, float] = Field(default_factory=dict)
    scenario: Optional[ScenarioSpec] = None
    unsafe_actions: int = 0
    remediated: bool = False
    context: dict[str, Any] = Field(default_factory=dict)


print("[OK] Types defined")

[OK] Types defined


In [24]:
# ============================================================================
# core/tools.py — Defender command surface
# ============================================================================

READ_ONLY_TOOLS = [
    "kubectl_get_pods", "kubectl_describe_pod", "kubectl_logs", "kubectl_top",
    "promql_query", "logql_query", "jaeger_search", "istioctl_proxy_status",
    "istioctl_routes", "curl_service", "dns_lookup", "check_deploy_history",
]
MUTATING_TOOLS = [
    "kubectl_rollout_undo", "kubectl_rollout_restart", "kubectl_scale",
    "feature_flag_toggle", "traffic_split_update", "kubectl_apply_config",
]
COMMUNICATION_TOOLS = ["post_status_update"]
TERMINAL_TOOLS = ["declare_resolved", "submit_rca"]
AVAILABLE_TOOLS = READ_ONLY_TOOLS + MUTATING_TOOLS + COMMUNICATION_TOOLS + TERMINAL_TOOLS


class ToolRuntime:
    def __init__(self, graph: MicroserviceGraph):
        self.graph = graph
        self.status_updates: list[str] = []
        self.unsafe_actions = 0
        self.resolved_declared = False
        self.submitted_rca: RCA | None = None

    def execute(self, command: str) -> str:
        parts = command.strip().split(maxsplit=1)
        if not parts:
            return "ERROR: empty command"
        tool = parts[0]
        args = parts[1] if len(parts) > 1 else ""
        handler = self._handlers().get(tool)
        if handler is None:
            return f"ERROR: unknown tool '{tool}'. Available tools: {', '.join(AVAILABLE_TOOLS)}"
        try:
            return handler(args)
        except KeyError as exc:
            return f"ERROR: {exc}"

    def _handlers(self) -> dict[str, Callable[[str], str]]:
        return {
            "kubectl_get_pods": self._get_pods,
            "kubectl_describe_pod": self._describe_pod,
            "kubectl_logs": self._logs,
            "kubectl_top": self._top,
            "promql_query": self._promql,
            "logql_query": self._logql,
            "jaeger_search": self._jaeger,
            "istioctl_proxy_status": self._proxy_status,
            "istioctl_routes": self._routes,
            "curl_service": self._curl,
            "dns_lookup": self._dns,
            "check_deploy_history": self._deploy_history,
            "kubectl_rollout_undo": lambda a: self._fix("kubectl_rollout_undo", a),
            "kubectl_rollout_restart": lambda a: self._fix("kubectl_rollout_restart", a),
            "kubectl_scale": lambda a: self._fix("kubectl_scale", a.split()[0] if a else a),
            "feature_flag_toggle": lambda a: self._fix("feature_flag_toggle", a.split()[0] if a else a),
            "traffic_split_update": lambda a: self._fix("traffic_split_update", a.split()[0] if a else a),
            "kubectl_apply_config": lambda a: self._fix("kubectl_apply_config", a.split()[0] if a else a),
            "post_status_update": self._status,
            "declare_resolved": self._declare,
            "submit_rca": self._submit_rca,
        }

    def _service_arg(self, args: str) -> str:
        return args.split()[0] if args.strip() else self.graph.root_cause_service

    def _get_pods(self, _: str) -> str:
        lines = ["NAME READY STATUS RESTARTS AGE"]
        for svc in self.graph.services.values():
            status = "Running" if not svc.current_fault else ("CrashLoopBackOff" if svc.current_fault == "oom_kill" else "Degraded")
            lines.append(f"{svc.name}-7d9c 1/1 {status} {3 if svc.current_fault else 0} 42m")
        return "\n".join(lines)

    def _describe_pod(self, args: str) -> str:
        svc = self.graph.get(self._service_arg(args))
        return f"Name: {svc.name}\nDependencies: {', '.join(svc.dependencies) or 'none'}\nFault: {svc.current_fault or 'none'}\nConfig: {json.dumps(svc.config, sort_keys=True)}"

    def _logs(self, args: str) -> str:
        return json_logs(self.graph, self._service_arg(args))

    def _top(self, args: str) -> str:
        svc = self.graph.get(self._service_arg(args))
        return f"NAME CPU% MEMORY% LATENCY_P99_MS ERROR_RATE\n{svc.name} {svc.cpu:.1f} {svc.memory:.1f} {svc.latency_p99:.0f} {svc.error_rate:.3f}"

    def _promql(self, args: str) -> str:
        svc_name = self._service_arg(args.replace("{", " ").replace("}", " ").replace("service=", " "))
        return prometheus_metrics(self.graph, svc_name if svc_name in self.graph.services else self.graph.root_cause_service)

    def _logql(self, args: str) -> str:
        return self._logs(args)

    def _jaeger(self, args: str) -> str:
        return jaeger_traces(self.graph, self._service_arg(args))

    def _proxy_status(self, _: str) -> str:
        return "\n".join(f"{s.name} SYNCED {'STALE' if s.current_fault else 'HEALTHY'}" for s in self.graph.services.values())

    def _routes(self, args: str) -> str:
        svc = self.graph.get(self._service_arg(args))
        return json.dumps({"service": svc.name, "routes": svc.dependencies, "envoy_flags": "UH UF URX NR" if svc.current_fault else ""})

    def _curl(self, args: str) -> str:
        svc = self.graph.get(self._service_arg(args))
        if svc.error_rate > 0.3 and not svc.remediated:
            return "HTTP/1.1 503 Service Unavailable\ncontext deadline exceeded"
        return "HTTP/1.1 200 OK\nok"

    def _dns(self, args: str) -> str:
        svc = self.graph.get(self._service_arg(args))
        if svc.current_fault == "dns_misconfig":
            return f"{svc.name}.svc.cluster.local NXDOMAIN"
        return f"{svc.name}.svc.cluster.local 10.43.12.7"

    def _deploy_history(self, args: str) -> str:
        return json.dumps(self.graph.get(self._service_arg(args)).deploy_history, indent=2)

    def _fix(self, tool: str, args: str) -> str:
        service = self._service_arg(args)
        correct = self.graph.mark_remediated(tool, service)
        if correct:
            return f"OK: {tool} applied to {service}; synthetic health checks are green"
        if service in {"postgres-primary", "redis-cache"} and tool in {"kubectl_rollout_restart", "kubectl_rollout_undo"}:
            self.unsafe_actions += 1
        return f"WARN: {tool} applied to {service}, but incident symptoms persist"

    def _status(self, args: str) -> str:
        self.status_updates.append(args)
        return "status update posted"

    def _declare(self, _: str) -> str:
        self.resolved_declared = True
        return "resolution declared; submit_rca is still required"

    def _submit_rca(self, args: str) -> str:
        try:
            self.submitted_rca = RCA.model_validate_json(args)
        except (ValidationError, ValueError) as exc:
            return f"ERROR: invalid RCA JSON: {exc}"
        return "RCA accepted"


print("[OK] ToolRuntime defined")

[OK] ToolRuntime defined


In [25]:
# ============================================================================
# Rubrics (standalone, no openenv dependency)
# ============================================================================

class Rubric(ABC):
    _rubric_children: Dict[str, "Rubric"]
    _forward_hooks: List[Callable]
    _forward_pre_hooks: List[Callable]
    last_score: Optional[float]

    def __init__(self):
        object.__setattr__(self, "_rubric_children", {})
        object.__setattr__(self, "_forward_hooks", [])
        object.__setattr__(self, "_forward_pre_hooks", [])
        object.__setattr__(self, "last_score", None)

    def __setattr__(self, name: str, value: Any) -> None:
        if isinstance(value, Rubric):
            self._rubric_children[name] = value
        object.__setattr__(self, name, value)

    def __call__(self, action: Any, observation: Any):
        for hook in self._forward_pre_hooks:
            hook(self, action, observation)
        result = self.forward(action, observation)
        self.last_score = result
        for hook in self._forward_hooks:
            hook(self, action, observation, result)
        return result

    @abstractmethod
    def forward(self, action: Any, observation: Any) -> float:
        raise NotImplementedError

    def children(self) -> Iterator["Rubric"]:
        yield from self._rubric_children.values()

    def named_children(self) -> Iterator[Tuple[str, "Rubric"]]:
        yield from self._rubric_children.items()

    def rubrics(self) -> Iterator["Rubric"]:
        for child in self._rubric_children.values():
            yield child
            yield from child.rubrics()

    def named_rubrics(self, prefix: str = "") -> Iterator[Tuple[str, "Rubric"]]:
        for name, child in self._rubric_children.items():
            full_name = f"{prefix}.{name}" if prefix else name
            yield full_name, child
            yield from child.named_rubrics(full_name)

    def reset(self) -> None:
        pass


class WeightedSum(Rubric):
    def __init__(self, rubrics: list[Rubric], weights: list[float]):
        super().__init__()
        if len(rubrics) != len(weights):
            raise ValueError(f"rubrics ({len(rubrics)}) != weights ({len(weights)})")
        if abs(sum(weights) - 1.0) > 1e-6:
            raise ValueError(f"Weights must sum to 1.0, got {sum(weights)}")
        for i, rubric in enumerate(rubrics):
            setattr(self, f"rubric_{i}", rubric)
        self._rubric_list = list(rubrics)
        self._weights = list(weights)

    def forward(self, action: Any, observation: Any) -> float:
        total = 0.0
        for rubric, weight in zip(self._rubric_list, self._weights):
            total += rubric(action, observation) * weight
        return total

    def __call__(self, action: Any, observation: Any):
        results = [rubric(action, observation) for rubric in self._rubric_list]
        for hook in self._forward_pre_hooks:
            hook(self, action, observation)
        total = sum(s * w for s, w in zip(results, self._weights))
        self.last_score = total
        for hook in self._forward_hooks:
            hook(self, action, observation, total)
        return total


class RecoveryRubric(Rubric):
    def forward(self, action, observation) -> float:
        runtime = observation.metadata.get("runtime")
        if runtime is None or not runtime.resolved_declared:
            return 0.0
        if not runtime.graph.is_recovered():
            return 0.0
        sr = runtime.graph.synthetic_success_rate()
        if sr >= 0.99:
            return 1.0
        if sr <= 0.5:
            return 0.0
        return (sr - 0.5) / 0.49


GENERIC_WHYS = {"unknown", "n/a", "todo", "because", "issue", "problem"}


class RCAQualityRubric(Rubric):
    def forward(self, action, observation) -> float:
        runtime = observation.metadata.get("runtime")
        if runtime is None or runtime.submitted_rca is None:
            return 0.0
        rca = runtime.submitted_rca
        graph = runtime.graph
        timeline_score = 1.0 if any(e.service == graph.root_cause_service for e in rca.timeline) else 0.0
        category_score = 1.0 if rca.root_cause_category == graph.root_cause_category else 0.0
        why_count = sum(1 for w in rca.five_whys if len(w.split()) >= 4 and w.strip().lower() not in GENERIC_WHYS)
        whys_score = min(1.0, why_count / 3.0)
        services = set(graph.service_names())
        action_score = 1.0 if any(any(s in item for s in services) for item in rca.action_items) else 0.0
        return 0.3 * timeline_score + 0.3 * category_score + 0.2 * whys_score + 0.2 * action_score


class BlastRadiusRubric(Rubric):
    def forward(self, action, observation) -> float:
        runtime = observation.metadata.get("runtime")
        if runtime is None:
            return 0.0
        return runtime.graph.blast_radius_score()


class SafetyRubric(Rubric):
    def forward(self, action, observation) -> float:
        runtime = observation.metadata.get("runtime")
        if runtime is None:
            return 1.0
        return max(0.0, 1.0 - runtime.unsafe_actions * 0.5)


def build_default_rubric() -> WeightedSum:
    return WeightedSum(
        [RecoveryRubric(), RCAQualityRubric(), BlastRadiusRubric(), SafetyRubric()],
        weights=[0.35, 0.30, 0.25, 0.10],
    )


print("[OK] Rubrics defined")

[OK] Rubrics defined


In [26]:
# ============================================================================
# scenario_compiler.py + env.py — Scenario compiler & Environment
# ============================================================================

def compile_scenario(spec: ScenarioSpec) -> MicroserviceGraph:
    graph = MicroserviceGraph.topology(spec.topology)
    inject_service = spec.inject_service if spec.inject_service in graph.services else "payment-service"
    FAULTS[spec.fault_primary].apply(graph, inject_service)
    graph.get(inject_service).latency_p99 = max(graph.get(inject_service).latency_p99, float(spec.latency_ms))
    if spec.fault_secondary:
        secondary_service = "redis-cache" if inject_service != "redis-cache" else "postgres-primary"
        FAULTS[spec.fault_secondary].apply(graph, secondary_service)
    if spec.red_herring:
        graph.get("user-service").logs.append(f"benign warning: {spec.red_herring}")
    if spec.deploy_window and spec.deploy_window != "none":
        graph.get(inject_service).deploy_history.append(
            {"version": "v3.3.0", "date": "2026-04-24", "status": "current", "notes": spec.deploy_window}
        )
    return graph


# Seed scenarios (inlined from YAML files)
SEED_SCENARIO_SPECS = [
    ScenarioSpec(task_id="seed_easy_memory_leak", topology="simple_fanout", fault_primary="oom_kill", inject_service="payment-service", latency_ms=4200, blast_radius=0.35, metric_noise=0.1, red_herring="stale_deploy_notice", deploy_window="recent_deploy", schema_drift="none", seed=7, max_steps=25),
    ScenarioSpec(task_id="seed_dns_misconfiguration", topology="mesh", fault_primary="dns_misconfig", inject_service="inventory-service", latency_ms=3600, blast_radius=0.45, metric_noise=0.2, red_herring="false_correlation", deploy_window="config_change", schema_drift="rename_metric", seed=11, max_steps=25),
    ScenarioSpec(task_id="seed_cert_expiry", topology="deep_chain", fault_primary="cert_expiry", inject_service="checkout-service", latency_ms=5000, blast_radius=0.7, metric_noise=0.1, red_herring="unrelated_alert", deploy_window="none", schema_drift="none", seed=17, max_steps=25),
    ScenarioSpec(task_id="seed_cache_stampede", topology="bipartite", fault_primary="cache_stampede", inject_service="redis-cache", latency_ms=3100, blast_radius=0.55, metric_noise=0.3, red_herring="old_anomaly", deploy_window="flag_flip", schema_drift="swap_units", seed=19, max_steps=25),
    ScenarioSpec(task_id="seed_replica_lag", topology="star", fault_primary="replica_lag", inject_service="postgres-primary", latency_ms=2100, blast_radius=0.4, metric_noise=0.25, red_herring="stale_deploy_notice", deploy_window="none", schema_drift="field_type_change", seed=23, max_steps=25),
    ScenarioSpec(task_id="seed_http_503_loop", topology="diamond", fault_primary="http_503_loop", fault_secondary="network_partition", inject_service="checkout-service", latency_ms=3900, blast_radius=0.8, metric_noise=0.35, red_herring="flapping_canary", deploy_window="recent_deploy", schema_drift="endpoint_version_bump", seed=29, max_steps=25),
]

DEFAULT_SCENARIO = SEED_SCENARIO_SPECS[0]


class OnCallRedShiftEnv:
    """Standalone three-agent SRE training environment."""

    def __init__(self, reward_cap: Optional[float] = None):
        self.rubric = build_default_rubric()
        self.reward_cap = reward_cap
        self._scenario = DEFAULT_SCENARIO
        self._runtime: ToolRuntime | None = None
        self._state = State()
        self._last_reward = 0.0

    def reset(self, seed: Optional[int] = None, episode_id: Optional[str] = None, task_id: Optional[str] = None, **kwargs: Any) -> Observation:
        if self.rubric is not None:
            self.rubric.reset()
        scenario_override = kwargs.get("scenario_spec")
        if scenario_override is not None:
            spec = scenario_override if isinstance(scenario_override, ScenarioSpec) else ScenarioSpec.model_validate(scenario_override)
        else:
            spec = self._load_scenario(task_id or episode_id or kwargs.get("task_id"))
        if seed is not None:
            spec = spec.model_copy(update={"seed": seed})
        self._scenario = spec
        self._runtime = ToolRuntime(compile_scenario(spec))
        self._state = State(episode_id=episode_id, task_id=spec.task_id, scenario=spec, context={"episode_id": episode_id})
        self._last_reward = 0.0
        return self._observation("Environment reset. You are the SRE responder. Investigate the alert, remediate safely, declare_resolved, then submit_rca.")

    def step(self, action: Action, timeout_s: Optional[float] = None, **kwargs: Any) -> Observation:
        if self._runtime is None:
            raise RuntimeError("reset() must be called before step()")
        if self._state.done:
            return self._observation("Episode already finished.")
        self._runtime.graph.tick(30)
        result = self._runtime.execute(action.command)
        self._state.step_count += 1
        self._state.actions_taken.append(action.command)
        self._state.last_action_result = result
        self._state.unsafe_actions = self._runtime.unsafe_actions
        self._state.resolved_declared = self._runtime.resolved_declared
        self._state.rca = self._runtime.submitted_rca
        self._state.remediated = self._runtime.graph.is_recovered()
        done = (
            self._state.step_count >= self._scenario.max_steps
            or (self._runtime.resolved_declared and self._runtime.submitted_rca is not None)
        )
        self._state.done = done
        obs = self._observation(result, done=done)
        reward = float(self.rubric(action, obs)) if self.rubric else 0.0
        if self.reward_cap is not None:
            reward = min(reward, self.reward_cap)
        self._last_reward = reward
        breakdown = self._reward_breakdown()
        self._state.reward_breakdown = breakdown
        obs.reward = reward
        obs.reward_breakdown = breakdown
        obs.metadata = {"task_id": self._scenario.task_id}
        return obs

    @property
    def state(self) -> State:
        return self._state

    def _observation(self, result: str, done: bool = False) -> Observation:
        runtime = self._runtime
        alerts: list[Alert] = []
        services: list[str] = []
        elapsed = 0
        if runtime is not None:
            services = runtime.graph.service_names()
            elapsed = runtime.graph.elapsed_sec
            alerts = [Alert(
                alert_id="ALT-RED-001", severity="critical",
                service=runtime.graph.root_cause_service,
                message=f"{runtime.graph.root_cause_category} symptoms detected with elevated p99/error rate",
                timestamp="2026-04-24T09:00:00Z",
            )]
        return Observation(
            done=done, reward=self._last_reward,
            metadata={"runtime": runtime} if runtime else {},
            alerts=alerts, last_action_result=result,
            available_tools=AVAILABLE_TOOLS, services=services,
            time_elapsed_sec=elapsed,
            goal="Restore customer-facing availability and submit a grounded RCA.",
            rca_required=True, reward_breakdown=self._state.reward_breakdown,
            task_id=self._scenario.task_id,
        )

    def _reward_breakdown(self) -> dict[str, float]:
        if self.rubric is None:
            return {}
        return {name: float(r.last_score or 0.0) for name, r in self.rubric.named_rubrics()}

    def _load_scenario(self, task_id: Optional[str]) -> ScenarioSpec:
        specs = {s.task_id: s for s in SEED_SCENARIO_SPECS}
        # Also load from curriculum buffer if it exists
        buf_path = os.path.join(os.getcwd(), "curriculum_results", "buffer.json")
        if os.path.exists(buf_path):
            try:
                buffer = RegretBuffer.load(Path(buf_path))
                for item in buffer.scenarios:
                    specs[item.spec.task_id] = item.spec
            except Exception:
                pass
        if task_id in specs:
            return specs[task_id]
        if task_id is None:
            return DEFAULT_SCENARIO
        raise ValueError(f"Unknown task_id {task_id}. Available: {', '.join(sorted(specs))}")


print("[OK] OnCallRedShiftEnv defined")

[OK] OnCallRedShiftEnv defined


In [27]:
# ============================================================================
# curriculum/buffer.py + mutator.py — Curriculum utilities
# ============================================================================

@dataclass
class BufferedScenario:
    spec: ScenarioSpec
    regret: float
    solve_rate: float
    seen_count: int = 0

    def to_dict(self) -> dict:
        return {"spec": self.spec.model_dump(), "regret": self.regret, "solve_rate": self.solve_rate, "seen_count": self.seen_count}

    @classmethod
    def from_dict(cls, payload: dict) -> "BufferedScenario":
        return cls(spec=ScenarioSpec.model_validate(payload["spec"]), regret=float(payload["regret"]), solve_rate=float(payload["solve_rate"]), seen_count=int(payload.get("seen_count", 0)))


class RegretBuffer:
    def __init__(self, scenarios: list[BufferedScenario] | None = None, epsilon: float = 0.08):
        self.scenarios = scenarios or []
        self.epsilon = epsilon

    def __len__(self) -> int:
        return len(self.scenarios)

    def add(self, item: BufferedScenario) -> None:
        self.scenarios.append(item)

    def sample(self, rng: random.Random) -> BufferedScenario:
        if not self.scenarios:
            raise ValueError("cannot sample from empty RegretBuffer")
        if rng.random() < self.epsilon:
            choice = rng.choice(self.scenarios)
        else:
            weights = [max(0.01, item.regret) for item in self.scenarios]
            choice = rng.choices(self.scenarios, weights=weights, k=1)[0]
        choice.seen_count += 1
        return choice

    def save(self, path: Path) -> None:
        path.parent.mkdir(parents=True, exist_ok=True)
        path.write_text(json.dumps([item.to_dict() for item in self.scenarios], indent=2), encoding="utf-8")

    @classmethod
    def load(cls, path: Path) -> "RegretBuffer":
        return cls([BufferedScenario.from_dict(item) for item in json.loads(path.read_text(encoding="utf-8"))])


MUTATOR_TOPOLOGIES = ["simple_fanout", "deep_chain", "mesh", "star", "bipartite", "diamond"]
MUTATOR_FAULTS = list(FAULTS.keys())
MUTATOR_SERVICES = ["api-gateway", "checkout-service", "payment-service", "inventory-service", "user-service", "postgres-primary", "redis-cache"]
MUTATOR_LATENCIES = [0, 50, 200, 1000, 5000]
MUTATOR_RED_HERRINGS = [None, "unrelated_alert", "stale_deploy_notice", "innocent_config_change", "flapping_canary", "false_correlation", "old_anomaly", "unused_service_spike"]
MUTATOR_DEPLOY_WINDOWS = ["recent_deploy", "flag_flip", "config_change", "none"]
MUTATOR_SCHEMA_DRIFTS = ["rename_metric", "swap_units", "rotate_creds", "new_required_field", "field_type_change", "endpoint_version_bump", "none"]


class ScenarioMutator:
    def __init__(self, rng: random.Random):
        self.rng = rng

    def mutate(self, parent: ScenarioSpec, generation: int) -> tuple[ScenarioSpec, str]:
        data = parent.model_dump()
        all_fields = ["topology", "fault_primary", "fault_secondary", "inject_service", "latency_ms", "blast_radius", "metric_noise", "red_herring", "deploy_window", "schema_drift"]
        fields = self.rng.sample(all_fields, k=1)
        for f in fields:
            self._apply_field_mutation(data, f)
        fingerprint = hashlib.sha1(repr(sorted(data.items())).encode()).hexdigest()[:10]
        data["task_id"] = f"evolved_{generation:04d}_{fingerprint}"
        data["seed"] = self.rng.randint(1, 2_000_000_000)
        data["max_steps"] = max(10, min(30, int(data.get("max_steps", 25))))
        return ScenarioSpec.model_validate(data), "+".join(fields)

    def _apply_field_mutation(self, data: dict, field: str) -> None:
        if field == "topology": data[field] = self.rng.choice(MUTATOR_TOPOLOGIES)
        elif field == "fault_primary": data[field] = self.rng.choice(MUTATOR_FAULTS)
        elif field == "fault_secondary": data[field] = self.rng.choice([None, *MUTATOR_FAULTS])
        elif field == "inject_service": data[field] = self.rng.choice(MUTATOR_SERVICES)
        elif field == "latency_ms": data[field] = self.rng.choice(MUTATOR_LATENCIES)
        elif field in {"blast_radius", "metric_noise"}: data[field] = round(min(1.0, max(0.0, float(data[field]) + self.rng.uniform(-0.2, 0.2))), 2)
        elif field == "red_herring": data[field] = self.rng.choice(MUTATOR_RED_HERRINGS)
        elif field == "deploy_window": data[field] = self.rng.choice(MUTATOR_DEPLOY_WINDOWS)
        elif field == "schema_drift": data[field] = self.rng.choice(MUTATOR_SCHEMA_DRIFTS)

    @staticmethod
    def novelty_key(spec: ScenarioSpec) -> tuple:
        return (spec.fault_primary, spec.fault_secondary, spec.inject_service, spec.schema_drift)


class RandomPolicyEvaluator:
    def __init__(self, rollout_count: int = 3, max_steps: int = 12, seed: int = 0):
        self.rollout_count = rollout_count
        self.max_steps = max_steps
        self._rng = random.Random(seed)

    def __call__(self, spec: ScenarioSpec) -> tuple[float, dict]:
        successes, rewards = [], []
        for _ in range(self.rollout_count):
            reward = self._run_rollout(spec)
            rewards.append(reward)
            successes.append(1 if reward >= 0.75 else 0)
        return sum(successes) / self.rollout_count, {"defender": "random_policy", "rollout_count": self.rollout_count, "successes": successes, "rewards": rewards}

    def _run_rollout(self, spec: ScenarioSpec) -> float:
        env = OnCallRedShiftEnv()
        obs = env.reset(task_id=spec.task_id, scenario_spec=spec)
        all_tools = list(READ_ONLY_TOOLS) + list(MUTATING_TOOLS)
        last_obs = obs
        for _ in range(self.max_steps):
            if last_obs.done:
                break
            tool = self._rng.choice(all_tools)
            service = self._rng.choice(last_obs.services) if last_obs.services else ""
            cmd = f"{tool} {service}".strip() if service else tool
            last_obs = env.step(Action(command=cmd))
        if not last_obs.done:
            last_obs = env.step(Action(command="declare_resolved"))
            rca_payload = json.dumps({"root_cause_service": obs.services[0] if obs.services else "unknown", "root_cause_category": "unknown", "timeline": [], "five_whys": ["Random policy rollout."], "action_items": [], "evidence_citations": [], "blast_radius_description": "Random policy evaluation."})
            last_obs = env.step(Action(command=f"submit_rca {rca_payload}"))
        return float(last_obs.reward or 0.0)


class AutocurriculumRunner:
    def __init__(self, seed_specs: list[ScenarioSpec], seed: int = 20260424, evaluator=None, max_buffer_size: int = 12):
        self.rng = random.Random(seed)
        self.mutator = ScenarioMutator(self.rng)
        self.max_buffer_size = max_buffer_size
        self.buffer = RegretBuffer([BufferedScenario(spec=spec, regret=0.5, solve_rate=0.5) for spec in seed_specs], epsilon=0.08)
        self.archive = {self.mutator.novelty_key(spec) for spec in seed_specs}
        self.evaluator = evaluator if evaluator is not None else RandomPolicyEvaluator(seed=seed)
        self.latest_rollout_stats: list[dict] = []
        self._prune_buffer()

    def _prune_buffer(self):
        if len(self.buffer) <= self.max_buffer_size:
            return
        seeds = [s for s in self.buffer.scenarios if not s.spec.task_id.startswith("evolved_")]
        evolved = sorted([s for s in self.buffer.scenarios if s.spec.task_id.startswith("evolved_")], key=lambda s: s.solve_rate * (1.0 - s.solve_rate), reverse=True)
        slots_for_evolved = max(0, self.max_buffer_size - len(seeds))
        before = len(self.buffer)
        self.buffer.scenarios = seeds + evolved[:slots_for_evolved]
        print(f"[CURRICULUM_PRUNE] buffer shrunk {before} -> {len(self.buffer)}")

    @classmethod
    def from_seed_specs(cls, seed_specs: list[ScenarioSpec], seed: int = 20260424, evaluator=None, max_buffer_size: int = 12):
        unique = {s.task_id: s for s in seed_specs}
        return cls(list(unique.values()), seed=seed, evaluator=evaluator, max_buffer_size=max_buffer_size)

    def evolve(self, iterations: int) -> RegretBuffer:
        generation, attempts = 0, 0
        added = evicted = skipped_novelty = skipped_variance = 0
        self._prune_buffer()
        buf_str = lambda: f"{len(self.buffer)}/{self.max_buffer_size}" if len(self.buffer) < self.max_buffer_size else f"{len(self.buffer)}"
        print(f"[CURRICULUM_EVOLVE] starting iterations={iterations} buffer={buf_str()}")
        while generation < iterations and attempts < iterations * 20:
            attempts += 1
            parent = self.buffer.sample(self.rng).spec
            candidate, mutated_field = self.mutator.mutate(parent, generation)
            novelty = self.mutator.novelty_key(candidate)
            if novelty in self.archive:
                skipped_novelty += 1
                continue
            solve_rate, eval_meta = self.evaluator(candidate)
            variance = solve_rate * (1.0 - solve_rate)
            regret = 1.0 - abs(0.5 - solve_rate) * 2.0
            new_item = BufferedScenario(spec=candidate, regret=regret, solve_rate=solve_rate)
            action = "add"
            if len(self.buffer) < self.max_buffer_size:
                self.buffer.add(new_item)
                added += 1
            else:
                evolved_items = [(i, item) for i, item in enumerate(self.buffer.scenarios) if item.spec.task_id.startswith("evolved_")]
                if evolved_items:
                    worst_idx, worst_item = min(evolved_items, key=lambda t: t[1].solve_rate * (1.0 - t[1].solve_rate))
                    if variance > worst_item.solve_rate * (1.0 - worst_item.solve_rate):
                        action = f"evict:{worst_item.spec.task_id}"
                        self.buffer.scenarios[worst_idx] = new_item
                        evicted += 1
                    else:
                        skipped_variance += 1
                        continue
                else:
                    self.buffer.add(new_item)
                    added += 1
            print(f"[CURRICULUM_MUTATE] gen={generation:04d} parent={parent.task_id} child={candidate.task_id} p={solve_rate:.3f} action={action}")
            self.archive.add(novelty)
            generation += 1
        print(f"[CURRICULUM_EVOLVE] done generations={generation} added={added} evicted={evicted}")
        return self.buffer


print("[OK] Curriculum system defined")

[OK] Curriculum system defined


In [28]:
# ============================================================================
# Quick sanity check: run one scenario end-to-end
# ============================================================================
env = OnCallRedShiftEnv()
obs = env.reset(task_id="seed_easy_memory_leak")
print(f"Task: {obs.task_id}")
print(f"Alert: {obs.alerts[0].message}")
print(f"Services: {obs.services}")
obs = env.step(Action(command="kubectl_logs payment-service"))
obs = env.step(Action(command="kubectl_rollout_restart payment-service"))
obs = env.step(Action(command="declare_resolved"))
rca = json.dumps({"root_cause_service": "payment-service", "root_cause_category": "oom_kill", "timeline": [{"timestamp": "2026-04-24T09:00:00Z", "service": "payment-service", "description": "OOM kill"}], "five_whys": ["payment-service ran out of memory", "Java heap space was exhausted", "Memory leak in payment processing"], "action_items": ["Add memory monitoring for payment-service"], "evidence_citations": [{"source": "log", "ref": "kubectl_logs payment-service", "excerpt": "OOMKilled"}], "blast_radius_description": "Payments degraded."})
obs = env.step(Action(command=f"submit_rca {rca}"))
print(f"Reward: {obs.reward:.3f}")
print(f"Done: {obs.done}")
print("[OK] Environment sanity check passed!")

Task: seed_easy_memory_leak
Alert: oom_kill symptoms detected with elevated p99/error rate
Services: ['api-gateway', 'checkout-service', 'payment-service', 'inventory-service', 'user-service', 'postgres-primary', 'redis-cache']
Reward: 0.988
Done: True
[OK] Environment sanity check passed!


## 4. Training Harness

All functions from `train_unsloth_grpo.py` inlined.

In [29]:
# ============================================================================
# Training harness (from train_unsloth_grpo.py)
# ============================================================================

SERVICES = ["api-gateway", "checkout-service", "payment-service", "inventory-service", "user-service", "postgres-primary", "redis-cache"]
SEED_TASKS = ["seed_easy_memory_leak", "seed_dns_misconfiguration", "seed_cert_expiry", "seed_cache_stampede", "seed_replica_lag", "seed_http_503_loop"]

COMMAND_RE = re.compile(
    r"\b(" + "|".join(re.escape(tool) for tool in [*READ_ONLY_TOOLS, *MUTATING_TOOLS, "declare_resolved"]) + r")\b(?:\s+([a-z0-9_.:/={}\"'-]+))?",
    re.IGNORECASE,
)

FAULT_RUNBOOK_HINTS = {
    "oom_kill": "memory pressure or OOMKilled usually needs kubectl_rollout_restart on the faulty service",
    "cpu_hog": "CPU saturation usually needs kubectl_scale on the faulty service",
    "network_partition": "network partition symptoms usually need traffic_split_update on the faulty service",
    "dns_misconfig": "DNS or no-route symptoms usually need kubectl_apply_config on the faulty service",
    "replica_lag": "replica lag usually needs feature_flag_toggle on the faulty service",
    "cache_stampede": "cache stampede usually needs feature_flag_toggle on the faulty service",
    "http_503_loop": "HTTP 503 loops usually need kubectl_rollout_undo on the faulty service",
    "deadlock": "deadlocks usually need kubectl_rollout_restart on the faulty service",
    "disk_full": "disk-full configuration incidents usually need kubectl_apply_config on the faulty service",
    "cert_expiry": "certificate expiry usually needs kubectl_apply_config on the faulty service",
    "clock_skew": "clock skew usually needs kubectl_rollout_restart on the faulty service",
    "gc_pause": "GC pause incidents usually need kubectl_rollout_restart on the faulty service",
}

PROMPT_TEMPLATES = ["standard", "runbook", "triage"]


def build_rca(service: str, category: str) -> str:
    return json.dumps({"root_cause_service": service, "root_cause_category": category, "timeline": [{"timestamp": "2026-04-24T09:00:00Z", "service": service, "description": f"{category} identified from Red Shift telemetry"}], "five_whys": [f"{service} emitted direct {category} symptoms.", "The failure propagated through dependent customer-facing services.", "The first mitigation needed to target the true faulty component."], "action_items": [f"Add regression alerting and runbook coverage for {service} {category}."], "evidence_citations": [{"source": "telemetry", "ref": f"kubectl_logs {service}", "excerpt": category}], "blast_radius_description": "Customer-facing requests saw elevated latency or errors before remediation."})


def required_pairs(required: list[str]) -> list[tuple[str, str]]:
    pairs = []
    for item in required:
        if ":" in item:
            tool, service = item.split(":", 1)
            pairs.append((tool, service))
    return pairs


def build_prompt(*, task_id, alert_service, alert_message, services, tools, root_service, root_category, required, prompt_mode, template) -> str:
    base = f"""You are the on-call SRE for OnCallEnv Red Shift.\n\nTask id: {task_id}\nCritical alert: {alert_service} reports {alert_message}\nAvailable services: {", ".join(services)}\nAvailable tools: {", ".join(tools)}\n\nReturn only the XML action block below. Put one simulator command per line inside:\n<actions>\n...\n</actions>\nStop immediately after the closing </actions> tag.\n\nUse real commands such as kubectl_logs SERVICE, promql_query SERVICE,\njaeger_search SERVICE, kubectl_rollout_restart SERVICE,\nkubectl_rollout_undo SERVICE, kubectl_scale SERVICE,\nfeature_flag_toggle SERVICE, traffic_split_update SERVICE,\nkubectl_apply_config SERVICE, and declare_resolved.\nDo not include explanations, markdown, bullets, JSON, RCA text, or prose outside the tags.\n"""
    if prompt_mode == "hard":
        return base
    accepted = ", ".join(f"{tool} {service}" for tool, service in required_pairs(required))
    hint = FAULT_RUNBOOK_HINTS.get(root_category, f"{root_category} symptoms should be remediated on {root_service}")
    if template == "runbook":
        return base + f"\nTraining runbook hint: suspected faulty service is {root_service}. Fault family is {root_category}. {hint}. Accepted remediation command for this easy curriculum item: {accepted}. A good answer is exactly 3-5 command lines and ends with </actions>.\n"
    if template == "triage":
        return base + f"\nEasy triage hints: first inspect {root_service}; then apply the remediation matching {root_category}; then declare_resolved. Gold remediation: {accepted}. A good answer is exactly 3-5 command lines and ends with </actions>.\n"
    return base + f"\nEasy-mode hints: root service = {root_service}; fault = {root_category}; best remediation = {accepted}. Include declare_resolved after the fix. A good answer is exactly 3-5 command lines and ends with </actions>.\n"


def inspect_task(task_id: str, *, prompt_mode: str = "hard", template: str = "standard") -> dict[str, Any]:
    env = OnCallRedShiftEnv()
    obs = env.reset(task_id=task_id)
    graph = env._runtime.graph
    service = graph.root_cause_service
    category = graph.root_cause_category
    required = sorted(graph.required_remediations)
    prompt = build_prompt(task_id=task_id, alert_service=obs.alerts[0].service, alert_message=obs.alerts[0].message, services=obs.services, tools=obs.available_tools, root_service=service, root_category=category, required=required, prompt_mode=prompt_mode, template=template)
    return {"prompt": prompt, "task_id": task_id, "root_service": service, "root_category": category, "required": required, "prompt_mode": prompt_mode, "template": template}


def load_task_ids(curriculum_buffer, max_tasks, seed, max_buffer_size=0):
    task_ids = list(SEED_TASKS)
    if curriculum_buffer and Path(curriculum_buffer).exists():
        buffer = RegretBuffer.load(Path(curriculum_buffer))
        if max_buffer_size > 0 and len(buffer.scenarios) > max_buffer_size:
            buffer.scenarios = buffer.scenarios[:max_buffer_size]
        task_ids = list(dict.fromkeys([item.spec.task_id for item in buffer.scenarios]))
    rng = random.Random(seed)
    rng.shuffle(task_ids)
    if max_tasks:
        task_ids = task_ids[:max_tasks]
    return task_ids


def extract_completion_text(completion) -> str:
    if isinstance(completion, str): return completion
    if isinstance(completion, list): return "\n".join(str(item.get("content", "")) if isinstance(item, dict) else str(item) for item in completion)
    if isinstance(completion, dict): return str(completion.get("content", completion))
    return str(completion)


def parse_commands(text: str, max_commands: int = 10) -> list[str]:
    match = re.search(r"<actions>(.*?)</actions>", text, flags=re.IGNORECASE | re.DOTALL)
    body = match.group(1) if match else text
    commands = []
    for raw_line in body.splitlines():
        line = raw_line.strip().strip("-*` ")
        if not line: continue
        found = COMMAND_RE.search(line)
        if not found: continue
        tool = found.group(1).lower()
        arg = (found.group(2) or "").strip().strip("'\"`")
        if tool in MUTATING_TOOLS or tool in {"kubectl_logs", "kubectl_top", "kubectl_describe_pod", "jaeger_search", "dns_lookup", "check_deploy_history", "curl_service", "promql_query", "logql_query", "istioctl_routes"}:
            service = next((svc for svc in SERVICES if svc in arg or svc in line), "")
            if not service: continue
            commands.append(f"{tool} {service}")
        elif tool == "declare_resolved":
            commands.append("declare_resolved")
        else:
            commands.append(tool)
        if len(commands) >= max_commands: break
    return commands


def shaped_easy_reward(completion, commands, env_reward, required, root_service) -> float:
    text = extract_completion_text(completion)
    lower = text.lower()
    pairs = required_pairs(required)
    required_tools = {tool for tool, _ in pairs}
    required_services = {service for _, service in pairs}
    command_set = set(commands)
    tools_seen = {c.split()[0] for c in commands if c.split()}
    services_seen = {svc for c in commands for svc in SERVICES if svc in c}
    score = 0.0
    if "<actions>" in lower and "</actions>" in lower: score += 0.10
    if commands: score += 0.08
    if any(c.split()[0] in READ_ONLY_TOOLS for c in commands if c.split()): score += 0.10
    if root_service in services_seen: score += 0.18
    elif required_services & services_seen: score += 0.12
    if required_tools & tools_seen: score += 0.18
    exact = sum(1 for t, s in pairs if f"{t} {s}" in command_set)
    if pairs: score += 0.28 * (exact / len(pairs))
    if "declare_resolved" in command_set: score += 0.06
    if 2 <= len(commands) <= 8: score += 0.04
    if any(c.startswith("submit_rca") for c in commands): score -= 0.05
    score = 0.75 * score + 0.25 * max(0.0, env_reward)
    return max(-0.1, min(1.0, score))


def rollout_reward(task_id, completion, root_service, root_category, required=None, reward_mode="hard", *, print_completions=False, verbose=False) -> float:
    text = extract_completion_text(completion)
    commands = parse_commands(text)
    if not commands: return -0.25
    env = OnCallRedShiftEnv()
    env.reset(task_id=task_id)
    obs = None
    for command in commands:
        obs = env.step(Action(command=command))
        if obs.done: break
    if not any(c == "declare_resolved" for c in commands):
        obs = env.step(Action(command="declare_resolved"))
    obs = env.step(Action(command=f"submit_rca {build_rca(root_service, root_category)}"))
    reward = float(obs.reward or 0.0)
    if reward_mode == "easy":
        return shaped_easy_reward(completion, commands, reward, required or [], root_service)
    format_bonus = 0.05 if "<actions>" in text.lower() and "</actions>" in text.lower() else 0.0
    concise_bonus = 0.03 if 2 <= len(commands) <= 8 else 0.0
    final_reward = max(-0.25, min(1.1, reward + format_bonus + concise_bonus))
    if print_completions:
        print(f"[DEFENDER_COMPLETION] task={task_id}")
        print(text)
        print(f"[PARSED_COMMANDS] {commands}")
        print(f"[ROLLOUT_REWARD] {final_reward:.4f}")
    return final_reward


def apply_feedback_to_buffer(path, feedback, *, verbose=False, max_buffer_size=0):
    if not feedback: return {"updated_tasks": 0, "added_tasks": 0}
    path = Path(path)
    buffer = RegretBuffer.load(path) if path.exists() else RegretBuffer([])
    if max_buffer_size > 0 and len(buffer.scenarios) > max_buffer_size:
        buffer.scenarios = buffer.scenarios[:max_buffer_size]
    by_task = {item.spec.task_id: item for item in buffer.scenarios}
    updated = 0
    new_solve_rates, all_rewards = [], []
    for task_id, rewards in feedback.items():
        if not rewards: continue
        all_rewards.extend(rewards)
        solve_rate = sum(1 for v in rewards if v >= 0.75) / len(rewards)
        regret = 1.0 - abs(0.5 - solve_rate) * 2.0
        item = by_task.get(task_id)
        if item is None: continue
        total_seen = max(1, item.seen_count + len(rewards))
        item.solve_rate = (item.solve_rate * item.seen_count + solve_rate * len(rewards)) / total_seen
        item.regret = (item.regret * item.seen_count + regret * len(rewards)) / total_seen
        item.seen_count = total_seen
        new_solve_rates.append(item.solve_rate)
        updated += 1
        if verbose:
            print(f"[CURRICULUM_FEEDBACK] task={task_id} solve_rate={item.solve_rate:.3f} regret={item.regret:.3f} seen={item.seen_count} rewards={[round(r, 3) for r in rewards]}")
    buffer.save(path)
    mean_reward = sum(all_rewards) / len(all_rewards) if all_rewards else 0.0
    mean_solve_rate = sum(new_solve_rates) / len(new_solve_rates) if new_solve_rates else 0.0
    print(f"[CURRICULUM_FEEDBACK] updated_tasks={updated} mean_reward={mean_reward:.4f} mean_solve_rate={mean_solve_rate:.3f} buffer_size={len(buffer)} path={path}")
    return {"updated_tasks": updated, "added_tasks": 0}


def evolve_curriculum(args, *, verbose=False):
    if args.curriculum_evolve_iterations <= 0:
        return {"evolved_count": 0}
    evaluator = RandomPolicyEvaluator(rollout_count=args.rollouts_per_candidate, max_steps=args.curriculum_defender_max_steps, seed=args.seed)
    runner = AutocurriculumRunner.from_seed_specs(SEED_SCENARIO_SPECS, seed=args.seed, evaluator=evaluator)
    if Path(args.curriculum_buffer).exists():
        existing = RegretBuffer.load(Path(args.curriculum_buffer))
        for item in existing.scenarios:
            if item.spec.task_id not in {s.spec.task_id for s in runner.buffer.scenarios}:
                runner.buffer.add(BufferedScenario(spec=item.spec, regret=item.regret, solve_rate=item.solve_rate, seen_count=item.seen_count))
                runner.archive.add(runner.mutator.novelty_key(item.spec))
    before = len(runner.buffer)
    buffer = runner.evolve(args.curriculum_evolve_iterations)
    if args.max_buffer_size > 0 and len(buffer.scenarios) > args.max_buffer_size:
        buffer.scenarios = buffer.scenarios[:args.max_buffer_size]
    buffer.save(Path(args.curriculum_buffer))
    if verbose:
        print(f"[CURRICULUM_EVOLVE] before={before} after={len(buffer.scenarios)}")
    return {"evolved_count": max(0, len(buffer.scenarios) - before)}


def evaluate_model(model, tokenizer, task_rows, out_path, max_new_tokens, reward_mode="hard"):
    import torch
    scores, generations = {}, {}
    model.eval()
    for row in task_rows:
        inputs = tokenizer(row["prompt"], return_tensors="pt").to(model.device)
        with torch.no_grad():
            output_ids = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False, pad_token_id=tokenizer.eos_token_id)
        completion = tokenizer.decode(output_ids[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True)
        reward = rollout_reward(row["task_id"], completion, row["root_service"], row["root_category"], required=row.get("required", []), reward_mode=reward_mode)
        scores[row["task_id"]] = reward
        generations[row["task_id"]] = {"completion": completion, "commands": parse_commands(completion), "reward": reward}
    summary = {"mean_reward": sum(scores.values()) / len(scores) if scores else 0.0, "scores": scores, "generations": generations}
    Path(out_path).write_text(json.dumps(summary, indent=2), encoding="utf-8")
    return summary


print("[OK] Training harness defined")

[OK] Training harness defined


## 5. GRPO Training Run

Configure and launch the Unsloth + TRL GRPO training loop.

In [30]:
import argparse
os.environ["WANDB_DISABLED"] = "true"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

# ==========================================
# Training configuration — edit these!
# ==========================================
class TrainArgs:
    model_name = "unsloth/Qwen2.5-3B-Instruct-bnb-4bit"
    out_dir = Path("training_results/unsloth_grpo_qwen3b_easy")
    curriculum_buffer = Path("curriculum_results/buffer.json")
    max_tasks = 120
    max_buffer_size = 12
    max_steps = 2
    per_device_train_batch_size = 2
    gradient_accumulation_steps = 8
    num_generations = 2
    max_seq_length = 1280
    max_prompt_length = 768
    max_completion_length = 256
    lr = 5e-6
    temperature = 0.8
    beta = 0.02
    scale_rewards = "batch"
    loss_type = "dr_grpo"
    reward_mode = "easy"
    prompt_mode = "easy"
    prompt_variants = 3
    logging_steps = 1
    save_steps = 50
    eval_tasks = 32
    lora_rank = 16
    lora_alpha = 32
    seed = 20260424
    print_completions = False
    verbose = True
    dynamic_curriculum = True
    curriculum_update_every = 40
    curriculum_evolve_iterations = 10
    curriculum_evolve_warmup_steps = 75
    curriculum_evolve_frequency = 2
    difficulty_source = "feedback"
    rollouts_per_candidate = 3
    curriculum_defender_model = None
    curriculum_defender_max_steps = 18
    curriculum_defender_temperature = 0.2
    resume_from_checkpoint = None

args = TrainArgs()
args.out_dir.mkdir(parents=True, exist_ok=True)
print(f"Output dir: {args.out_dir}")
print(f"Model: {args.model_name}")
print(f"Max steps: {args.max_steps}")

Output dir: training_results/unsloth_grpo_qwen3b_easy
Model: unsloth/Qwen2.5-3B-Instruct-bnb-4bit
Max steps: 2


In [31]:
# ==========================================
# Build curriculum dataset
# ==========================================
start_time = time.time()

task_ids = load_task_ids(args.curriculum_buffer, args.max_tasks, args.seed, args.max_buffer_size)
rows = []
for tid in task_ids:
    for idx in range(max(1, args.prompt_variants)):
        template = PROMPT_TEMPLATES[idx % len(PROMPT_TEMPLATES)]
        rows.append(inspect_task(tid, prompt_mode=args.prompt_mode, template=template))
random.Random(args.seed).shuffle(rows)
eval_rows = rows[:min(args.eval_tasks, len(rows))]

dataset_path = args.out_dir / "dataset.jsonl"
dataset_path.write_text("\n".join(json.dumps(row) for row in rows) + "\n", encoding="utf-8")

print(f"Training tasks: {len(rows)}")
print(f"Eval tasks: {len(eval_rows)}")
print(f"Unique task IDs: {len(task_ids)}")

Training tasks: 18
Eval tasks: 18
Unique task IDs: 6


In [32]:
# ==========================================
# Load model with Unsloth + LoRA
# ==========================================
from datasets import Dataset
from unsloth import FastLanguageModel, PatchFastRL, is_bfloat16_supported

try:
    PatchFastRL("GRPO", FastLanguageModel)
except Exception as exc:
    print(f"WARN: PatchFastRL('GRPO') failed or was already applied: {exc}")
from trl import GRPOTrainer, GRPOConfig

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=args.model_name,
    max_seq_length=args.max_seq_length,
    load_in_4bit=True,
    fast_inference=False,
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = FastLanguageModel.get_peft_model(
    model,
    r=args.lora_rank,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=args.lora_alpha,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=args.seed,
)
model.train()

# Verify LoRA is trainable
total, trainable, lora_count = 0, 0, 0
for name, p in model.named_parameters():
    total += p.numel()
    if p.requires_grad: trainable += p.numel()
    if "lora" in name.lower(): lora_count += 1
if trainable == 0:
    for name, p in model.named_parameters():
        if "lora" in name.lower(): p.requires_grad_(True)

if hasattr(model, "print_trainable_parameters"):
    model.print_trainable_parameters()
print(f"Trainable: {trainable:,} / {total:,} | LoRA tensors: {lora_count}")

==((====))==  Unsloth 2025.11.1: Fast Qwen2 patching. Transformers: 4.57.2.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
trainable params: 29,933,568 || all params: 3,115,872,256 || trainable%: 0.9607
Trainable: 29,933,568 / 1,728,606,208 | LoRA tensors: 504


In [33]:
# ==========================================
# Run baseline evaluation
# ==========================================
baseline = evaluate_model(model, tokenizer, eval_rows, args.out_dir / "baseline_generations.json", args.max_completion_length, args.reward_mode)
print(f"Baseline mean reward: {baseline['mean_reward']:.4f}")

Baseline mean reward: 0.5211


In [35]:
# ==========================================
# GRPO Training Loop
# ==========================================
reward_feedback: dict[str, list[float]] = defaultdict(list)
reward_counter = {"count": 0}
step_counter = {"count": 0}
curriculum_update_count = {"n": 0}
curriculum_updates: list[dict[str, Any]] = []

def redshift_reward(completions, task_id, root_service, root_category, required, **kwargs):
    rewards = []
    for completion, tid, svc, cat, req in zip(completions, task_id, root_service, root_category, required):
        reward = rollout_reward(tid, completion, svc, cat, required=req, reward_mode=args.reward_mode, print_completions=args.print_completions, verbose=args.verbose)
        rewards.append(reward)
        if args.dynamic_curriculum:
            reward_feedback[tid].append(reward)
            reward_counter["count"] += 1
    step_counter["count"] += 1
    if args.dynamic_curriculum and reward_counter["count"] >= max(1, args.curriculum_update_every):
        update_info = apply_feedback_to_buffer(args.curriculum_buffer, dict(reward_feedback), verbose=args.verbose, max_buffer_size=args.max_buffer_size)
        reward_feedback.clear()
        reward_counter["count"] = 0
        curriculum_update_count["n"] += 1
        n = curriculum_update_count["n"]
        step = step_counter["count"]
        warmup_done = step >= args.curriculum_evolve_warmup_steps
        on_cycle = (n % max(1, args.curriculum_evolve_frequency)) == 0
        if args.curriculum_evolve_iterations > 0 and warmup_done and on_cycle:
            print(f"[CURRICULUM_GATE] step={step} update={n} -> evolving")
            update_info.update(evolve_curriculum(args, verbose=args.verbose))
        elif args.curriculum_evolve_iterations > 0:
            reason = "warmup" if not warmup_done else f"freq (every {args.curriculum_evolve_frequency})"
            print(f"[CURRICULUM_GATE] step={step} update={n} -> skipping evolution ({reason})")
        curriculum_updates.append(update_info)
    return rewards


train_dataset = Dataset.from_list(rows)

grpo_kwargs = {
    "output_dir": str(args.out_dir),
    "learning_rate": args.lr,
    "per_device_train_batch_size": args.per_device_train_batch_size,
    "gradient_accumulation_steps": args.gradient_accumulation_steps,
    "num_generations": args.num_generations,
    "max_prompt_length": args.max_prompt_length,
    "max_completion_length": args.max_completion_length,
    "max_steps": args.max_steps,
    "temperature": args.temperature,
    "logging_steps": args.logging_steps,
    "save_steps": args.save_steps,
    "report_to": [],
    "remove_unused_columns": False,
    "fp16": True,
    "bf16": False,
    "seed": args.seed,
}
for optional_key, value in {"beta": args.beta, "scale_rewards": args.scale_rewards, "loss_type": args.loss_type, "use_vllm": False}.items():
    try:
        test = dict(grpo_kwargs)
        test[optional_key] = value
        GRPOConfig(**test)
        grpo_kwargs[optional_key] = value
    except TypeError:
        continue

training_args = GRPOConfig(**grpo_kwargs)

trainer_kwargs = {"model": model, "reward_funcs": redshift_reward, "args": training_args, "train_dataset": train_dataset}
trainer_params = inspect.signature(GRPOTrainer.__init__).parameters
if "processing_class" in trainer_params:
    trainer_kwargs["processing_class"] = tokenizer
else:
    trainer_kwargs["tokenizer"] = tokenizer

trainer = GRPOTrainer(**trainer_kwargs)
trainer.train(resume_from_checkpoint=str(args.resume_from_checkpoint) if args.resume_from_checkpoint else None)

The model is already on multiple devices. Skipping the move to device specified in `args`.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 18 | Num Epochs = 1 | Total steps = 2
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 8
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 8 x 1) = 16
 "-____-"     Trainable parameters = 29,933,568 of 3,115,872,256 (0.96% trained)


Step,Training Loss,reward,reward_std,completions / mean_length,completions / min_length,completions / max_length,completions / clipped_ratio,completions / mean_terminated_length,completions / min_terminated_length,completions / max_terminated_length,sampling / sampling_logp_difference / mean,sampling / sampling_logp_difference / max,sampling / importance_sampling_ratio / min,sampling / importance_sampling_ratio / mean,sampling / importance_sampling_ratio / max,kl,rewards / redshift_reward / mean,rewards / redshift_reward / std
1,0.000000,0.661879,0.270191,256.000000,256.000000,256.000000,1.000000,0.000000,0.000000,0.000000,0,0,0,0,0,0.000019,0.661880,0.270191
2,0.000000,0.633787,0.259214,256.000000,256.000000,256.000000,1.000000,0.000000,0.000000,0.000000,No Log,No Log,No Log,No Log,No Log,0.000031,0.633787,0.259214


Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).


TrainOutput(global_step=2, training_loss=5.21540641784668e-07, metrics={'train_runtime': 144.8354, 'train_samples_per_second': 0.221, 'train_steps_per_second': 0.014, 'total_flos': 0.0, 'train_loss': 5.21540641784668e-07})

In [ ]:
# ==========================================
# Save adapter + trained evaluation
# ==========================================
adapter_dir = args.out_dir / "adapter"
model.save_pretrained(adapter_dir)
tokenizer.save_pretrained(adapter_dir)

trained = evaluate_model(model, tokenizer, eval_rows, args.out_dir / "trained_generations.json", args.max_completion_length, reward_mode=args.reward_mode)

if args.dynamic_curriculum and reward_feedback:
    update_info = apply_feedback_to_buffer(args.curriculum_buffer, dict(reward_feedback), verbose=args.verbose, max_buffer_size=args.max_buffer_size)
    curriculum_updates.append(update_info)

summary = {
    "model_name": args.model_name,
    "duration_sec": time.time() - start_time,
    "max_steps": args.max_steps,
    "num_train_tasks": len(rows),
    "num_eval_tasks": len(eval_rows),
    "reward_mode": args.reward_mode,
    "prompt_mode": args.prompt_mode,
    "baseline_mean_reward": baseline["mean_reward"],
    "trained_mean_reward": trained["mean_reward"],
    "adapter_dir": str(adapter_dir),
    "curriculum_updates": curriculum_updates,
}
(args.out_dir / "summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
print(json.dumps(summary, indent=2))

## 6. Training Reward Curve

In [ ]:
import matplotlib.pyplot as plt

run_dir = Path("training_results/unsloth_grpo_qwen3b_easy")
checkpoints = sorted(run_dir.glob("checkpoint-*"), key=lambda p: int(p.name.split("-")[1]))

if not checkpoints:
    print("No checkpoints found yet!")
else:
    latest = checkpoints[-1]
    state_path = latest / "trainer_state.json"
    if not state_path.exists():
        print(f"Missing {state_path}")
    else:
        state = json.loads(state_path.read_text())
        log_history = state.get("log_history", [])
        steps, rewards = [], []
        for row in log_history:
            reward = row.get("reward", row.get("rewards/redshift_reward/mean"))
            if reward is not None:
                steps.append(row.get("step"))
                rewards.append(reward)
        if not rewards:
            print("No reward logs found yet.")
        else:
            plt.figure(figsize=(10, 5), dpi=120)
            plt.plot(steps, rewards, marker="x", linewidth=2, color="#1f77b4", label="Reward")
            plt.axhline(max(rewards), color="#2E7D32", linestyle="--", linewidth=1.5, label=f"Best: {max(rewards):.2f}")
            plt.title("Easy Qwen2.5 3B GRPO Reward Curve", fontsize=14)
            plt.xlabel("Training Step", fontsize=12)
            plt.ylabel("Reward Score", fontsize=12)
            plt.ylim(0, max(1.0, max(rewards) + 0.1))
            plt.legend()
            plt.grid(alpha=0.3)
            plt.show()

## 7. Model Comparison & Ablation Plots

Side-by-side evaluation of different checkpoints/models.

In [ ]:
import torch
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from peft import PeftModel
from unsloth import FastLanguageModel
from tqdm.auto import tqdm
import textwrap

sns.set_theme(style="whitegrid")
plt.rcParams['figure.dpi'] = 140

# ==========================================
# Configure models to compare
# ==========================================
MODEL_DICT = {
    "Feedback": "training_results/unsloth_grpo_qwen3b_easy/checkpoint-300",
    # Add more models here, e.g.:
    # "No Feedback": "/path/to/another/checkpoint",
}
models_to_test = {name: path for name, path in MODEL_DICT.items() if path and Path(path).exists()}

dataset_path = Path("training_results/unsloth_grpo_qwen3b_easy/dataset.jsonl")
eval_rows = [json.loads(line) for line in dataset_path.read_text().splitlines() if line.strip()][:24]
print(f"Loaded {len(eval_rows)} evaluation routines.")
print(f"Models to test: {list(models_to_test.keys())}")

In [ ]:
# ==========================================
# Run the ablation experiment
# ==========================================
all_results = []
BASE_MODEL_NAME = "unsloth/Qwen2.5-3B-Instruct-bnb-4bit"

for model_name, adapter_path in models_to_test.items():
    print(f"\n[+] Cold-booting base model for '{model_name}'...")
    base_model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=BASE_MODEL_NAME, max_seq_length=1280, load_in_4bit=True, fast_inference=False,
    )
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    print(f"[-] Mounting adapter: {adapter_path}...")
    active_model = PeftModel.from_pretrained(base_model, str(adapter_path), is_trainable=False)
    FastLanguageModel.for_inference(active_model)
    active_model.eval()

    for eval_idx, row in tqdm(enumerate(eval_rows), desc=f"Scoring {model_name}", total=len(eval_rows)):
        inputs = tokenizer(row["prompt"], return_tensors="pt").to(active_model.device)
        with torch.no_grad():
            output_ids = active_model.generate(**inputs, max_new_tokens=256, do_sample=False, pad_token_id=tokenizer.eos_token_id)
        completion = tokenizer.decode(output_ids[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True)
        reward = rollout_reward(
            task_id=row["task_id"], completion=completion, root_service=row["root_service"],
            root_category=row["root_category"], required=row.get("required", []), reward_mode="easy"
        )
        unique_id = f"Run {eval_idx+1:02d}: {row['task_id'][:15]}.."
        all_results.append({"Model": model_name, "Unique_Task_ID": unique_id, "Reward": reward})

    print(f"Wiping {model_name} from GPU memory...")
    del active_model, base_model, tokenizer
    torch.cuda.empty_cache()

df = pd.DataFrame(all_results)
mean_scores = df.groupby("Model")["Reward"].mean().reset_index()
print("\n" + "="*40 + "\nOVERALL MEAN REWARDS\n" + "="*40)
display(mean_scores)

In [ ]:
# ==========================================
# Ablation Plots
# ==========================================
if len(models_to_test) >= 1:
    plt.figure(figsize=(7, 5))
    ax = sns.barplot(data=mean_scores, x="Model", y="Reward", palette="viridis")
    plt.title("Ablation: Impact of Defender Feedback Curriculum", fontsize=14, pad=15, fontweight="bold")
    plt.ylabel("Average Reward Score", fontsize=12)
    plt.ylim(0, max(mean_scores["Reward"]) + 0.15)
    for p in ax.patches:
        ax.annotate(f"{p.get_height():.3f}", (p.get_x() + p.get_width() / 2., p.get_height()), ha='center', va='bottom', fontsize=12, xytext=(0, 5), textcoords='offset points')
    plt.show()

if len(models_to_test) >= 1:
    plt.figure(figsize=(8, 4.5))
    sns.boxplot(data=df, x="Reward", y="Model", width=0.5)
    sns.stripplot(data=df, x="Reward", y="Model", color=".2", size=5, alpha=0.6)
    plt.title("Score Consistency & Variance", fontsize=13, pad=12)
    plt.xlabel("Individual Task Reward", fontsize=12)
    plt.ylabel("")
    plt.show()

if len(models_to_test) >= 2:
    df_pivot = df.pivot(index="Unique_Task_ID", columns="Model", values="Reward").reset_index()
    m_feedback = list(MODEL_DICT.keys())[0]
    m_baseline = list(MODEL_DICT.keys())[1]
    df_pivot.sort_values(by=m_feedback, ascending=True, inplace=True)
    plt.figure(figsize=(9, 9))
    plt.scatter(df_pivot[m_feedback], df_pivot["Unique_Task_ID"], color="#27AE60", label=m_feedback, alpha=1.0, s=90, zorder=3)
    plt.scatter(df_pivot[m_baseline], df_pivot["Unique_Task_ID"], color="#7F8C8D", label=m_baseline, alpha=0.8, s=70, zorder=3)
    for i, row in df_pivot.iterrows():
        plt.plot([row[m_baseline], row[m_feedback]], [row["Unique_Task_ID"], row["Unique_Task_ID"]], color="gray", linestyle="--", alpha=0.4, zorder=1)
    plt.title("Task-by-Task Improvement Gap", fontsize=14, pad=15, fontweight="bold")
    plt.xlabel("Reward Score", fontsize=12)
    plt.xlim(-0.05, 1.05)
    plt.ylabel("")
    plt.legend(loc="upper left")
    plt.grid(axis='y', alpha=0.2)
    plt.show()

## 8. Single Task Demo

Show what the trained model actually outputs for a specific incident.

In [ ]:
import torch
import textwrap
from peft import PeftModel
from unsloth import FastLanguageModel

# ==========================================
# Configure which models / task to demo
# ==========================================
DEMO_MODELS = {
    "Feedback": "training_results/unsloth_grpo_qwen3b_easy/checkpoint-300",
    # "No Feedback": "/path/to/other/checkpoint",
}
demo_models = {n: p for n, p in DEMO_MODELS.items() if p and Path(p).exists()}

TARGET_TASK = "seed_easy_memory_leak"
task_data = inspect_task(TARGET_TASK, prompt_mode="hard", template="standard")

print("INCOMING SRE INCIDENT ALERT (THE PROMPT)")
print("=" * 80)
print(task_data["prompt"])
print("=" * 80 + "\n")

if demo_models:
    print("Loading Base Model...\n")
    BASE_MODEL_NAME = "unsloth/Qwen2.5-3B-Instruct-bnb-4bit"
    base_model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=BASE_MODEL_NAME, max_seq_length=1536, load_in_4bit=True, fast_inference=False,
    )
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    inputs = tokenizer(task_data["prompt"], return_tensors="pt").to(base_model.device)

    for model_name, adapter_path in demo_models.items():
        print(f"TESTING MODEL: {model_name}")
        print("-" * 50)
        active_model = PeftModel.from_pretrained(base_model, str(adapter_path), is_trainable=False)
        FastLanguageModel.for_inference(active_model)
        active_model.eval()
        with torch.no_grad():
            output_ids = active_model.generate(**inputs, max_new_tokens=256, do_sample=False, pad_token_id=tokenizer.eos_token_id)
        completion = tokenizer.decode(output_ids[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True)
        reward = rollout_reward(
            task_id=task_data["task_id"], completion=completion, root_service=task_data["root_service"],
            root_category=task_data["root_category"], required=task_data.get("required", []), reward_mode="easy"
        )
        print(">>> WHAT THE MODEL DECIDED TO DO:")
        print(textwrap.indent(completion.strip(), "    "))
        print(f"\nFINAL REWARD SCORE: {reward:.3f} / 1.000\n\n")
        del active_model
        torch.cuda.empty_cache()
else:
    print("No model checkpoints found to demo. Run training first!")

## 9. Archive Outputs

In [ ]:
import shutil

result_dirs = list(Path("training_results").glob("unsloth_grpo_qwen3b_*"))
if result_dirs:
    archive_path = "qwen3b_grpo_results.tar.gz"
    !tar -czf {archive_path} {' '.join(str(d) for d in result_dirs)}
    print(f"Archived {len(result_dirs)} result dirs to {archive_path}")
    !ls -lh {archive_path}
else:
    print("No result directories to archive.")